In [1]:
! Julia fractal - Fortran translation
! Original: K. Moerman 2026 (BASIC dialect)
!
! Classic escape-time Julia set renderer (fixed c = 0.357 + 0.357i),
! using a precomputed 512-entry colour palette and 4-fold (2-way here,
! via the w-x/h-y mirror used in the original) symmetry to halve the
! work. Output is written directly as a real PNG file using the same
! from-scratch zlib/DEFLATE (stored blocks) + CRC32/Adler32 encoder
! used for the Barnsley fern.

program julia_fractal
  implicit none

  integer, parameter :: w = 600, h = 700
  integer, parameter :: hw = w / 2, hh = h / 2
  integer, parameter :: N = 512
  real(kind=8), parameter :: Cre = 0.357d0, Cim = 0.357d0

  integer :: pal_r(0:N-1), pal_g(0:N-1), pal_b(0:N-1)
  integer, allocatable :: img_r(:,:), img_g(:,:), img_b(:,:)   ! img_*(x,y), 1-based
  integer :: x, y, c
  real(kind=8) :: Zre, Zim, Zre_sq, Zim_sq, Zre_start

  allocate(img_r(w, h), img_g(w, h), img_b(w, h))
  img_r = 0
  img_g = 0
  img_b = 0

  call calccolors(N, pal_r, pal_g, pal_b)

  do x = 0, hw
     Zre_start = real(x - hw, 8) / real(hw, 8)
     do y = 0, h
        Zre = Zre_start
        Zim = real(y - hh, 8) / real(hh, 8) * 1.3d0
        c = 0
        do
           Zre_sq = Zre * Zre
           Zim_sq = Zim * Zim
           Zim = 2.0d0 * Zre * Zim + Cim
           Zre = Zre_sq - Zim_sq + Cre
           c = c + 1
           if (c > N - 2 .or. (Zre_sq + Zim_sq) > 4.0d0) exit
        end do

        call setpixel(img_r, img_g, img_b, w, h, x,     y,     pal_r(c), pal_g(c), pal_b(c))
        call setpixel(img_r, img_g, img_b, w, h, w - x, h - y, pal_r(c), pal_g(c), pal_b(c))
     end do
  end do

  call save_png_rgb('julia256.png', img_r, img_g, img_b, w, h)
  print *, 'Saved julia256.png (', w, 'x', h, ')'

contains

  ! pre-calculate the N-entry RGB palette (faster than computing per pixel)
  subroutine calccolors(N, pal_r, pal_g, pal_b)
    integer, intent(in) :: N
    integer, intent(out) :: pal_r(0:N-1), pal_g(0:N-1), pal_b(0:N-1)
    integer :: k, ic
    real(kind=8) :: cval
    do k = 0, N - 1
       cval = 255.0d0 * sqrt(real(k, 8)) / sqrt(real(N, 8))
       ic = int(cval)
       pal_r(k) = mod(ic, 32) * 8
       pal_g(k) = mod(ic, 128) * 2
       pal_b(k) = mod(ic, 64) * 4
    end do
  end subroutine calccolors

  ! write one pixel if (x,y) - given in the original's 0-based coords -
  ! falls inside the w x h canvas; silently clipped otherwise (mirrors
  ! how the original BASIC graphics library clips off-canvas plot calls)
  subroutine setpixel(img_r, img_g, img_b, w, h, x, y, cr, cg, cb)
    integer, intent(in) :: w, h, x, y, cr, cg, cb
    integer, intent(inout) :: img_r(w, h), img_g(w, h), img_b(w, h)
    if (x >= 0 .and. x <= w - 1 .and. y >= 0 .and. y <= h - 1) then
       img_r(x + 1, y + 1) = cr
       img_g(x + 1, y + 1) = cg
       img_b(x + 1, y + 1) = cb
    end if
  end subroutine setpixel

  !=========================================================================
  ! Minimal PNG writer (no zlib/libpng dependency), RGB triples supplied
  ! as three separate channel arrays.
  !=========================================================================

  subroutine save_png_rgb(filename, img_r, img_g, img_b, w, h)
    character(len=*), intent(in) :: filename
    integer, intent(in) :: w, h
    integer, intent(in) :: img_r(w, h), img_g(w, h), img_b(w, h)
    integer :: iu
    integer(kind=8) :: crc_table(0:255)
    integer(kind=8) :: raw_size, nfull, rem, nblocks, deflate_size, idat_len
    integer(kind=8) :: crc, adler_a, adler_b, remaining_in_block, total_remaining
    integer :: i, j
    character(len=13) :: ihdr_data

    call build_crc_table(crc_table)

    open(newunit=iu, file=filename, access='stream', form='unformatted', status='replace')

    ! --- PNG signature ---------------------------------------------------
    write(iu) achar(137), achar(80), achar(78), achar(71), &
               achar(13), achar(10), achar(26), achar(10)

    ! --- IHDR chunk --------------------------------------------------------
    ihdr_data = char_be32(w) // char_be32(h) // achar(8) // achar(2) // &
                achar(0) // achar(0) // achar(0)
    call write_chunk(iu, 'IHDR', ihdr_data, 13, crc_table)

    ! --- IDAT chunk (streamed) ----------------------------------------------
    raw_size = int(h, 8) * int(1 + 3 * w, 8)
    nfull = raw_size / 65535_8
    rem   = raw_size - nfull * 65535_8
    if (rem > 0_8) then
       nblocks = nfull + 1_8
    else
       nblocks = nfull
    end if
    deflate_size = nblocks * 5_8 + raw_size
    idat_len = 2_8 + deflate_size + 4_8

    write(iu) char_be32_8(idat_len)
    write(iu) 'IDAT'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IDAT')

    call emit_byte(iu, 120, crc, crc_table)   ! zlib CMF = 0x78
    call emit_byte(iu, 1,   crc, crc_table)   ! zlib FLG = 0x01

    adler_a = 1_8
    adler_b = 0_8
    remaining_in_block = 0_8
    total_remaining = raw_size

    do j = 1, h
       call emit_raw_byte(iu, 0, crc, crc_table, adler_a, adler_b, &
                           remaining_in_block, total_remaining)   ! filter type: None
       do i = 1, w
          call emit_raw_byte(iu, img_r(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
          call emit_raw_byte(iu, img_g(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
          call emit_raw_byte(iu, img_b(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
       end do
    end do

    call emit_byte(iu, int(iand(ishft(adler_b, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_b, 255_8)),            crc, crc_table)
    call emit_byte(iu, int(iand(ishft(adler_a, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_a, 255_8)),            crc, crc_table)

    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    ! --- IEND chunk -----------------------------------------------------------
    write(iu) char_be32(0)
    write(iu) 'IEND'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IEND')
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    close(iu)
  end subroutine save_png_rgb

  subroutine emit_byte(iu, byteval, crc, crc_table)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    write(iu) achar(byteval)
    crc = ieor(crc_table(iand(ieor(crc, int(byteval, 8)), 255_8)), ishft(crc, -8))
  end subroutine emit_byte

  subroutine emit_raw_byte(iu, byteval, crc, crc_table, adler_a, adler_b, &
                            remaining_in_block, total_remaining)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc, adler_a, adler_b
    integer(kind=8), intent(inout) :: remaining_in_block, total_remaining
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: block_len, nlen
    logical :: is_last

    if (remaining_in_block == 0_8) then
       block_len = min(65535_8, total_remaining)
       is_last = (total_remaining <= 65535_8)
       call emit_byte(iu, merge(1, 0, is_last), crc, crc_table)
       nlen = 65535_8 - block_len
       call emit_byte(iu, int(iand(block_len, 255_8)),            crc, crc_table)
       call emit_byte(iu, int(iand(ishft(block_len, -8), 255_8)), crc, crc_table)
       call emit_byte(iu, int(iand(nlen, 255_8)),                 crc, crc_table)
       call emit_byte(iu, int(iand(ishft(nlen, -8), 255_8)),      crc, crc_table)
       remaining_in_block = block_len
    end if

    call emit_byte(iu, byteval, crc, crc_table)
    adler_a = mod(adler_a + int(byteval, 8), 65521_8)
    adler_b = mod(adler_b + adler_a, 65521_8)

    remaining_in_block = remaining_in_block - 1_8
    total_remaining = total_remaining - 1_8
  end subroutine emit_raw_byte

  subroutine write_chunk(iu, ctype, data, dlen, crc_table)
    integer, intent(in) :: iu, dlen
    character(len=*), intent(in) :: ctype
    character(len=*), intent(in) :: data
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: crc

    write(iu) char_be32(dlen)
    write(iu) ctype
    write(iu) data(1:dlen)

    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, ctype)
    call crc_update_bytes(crc, crc_table, data(1:dlen))
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)
  end subroutine write_chunk

  subroutine crc_update_bytes(crc, crc_table, s)
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    character(len=*), intent(in) :: s
    integer :: k
    do k = 1, len(s)
      crc = ieor(crc_table(iand(ieor(crc, int(iachar(s(k:k)), 8)), 255_8)), ishft(crc, -8))
    end do
  end subroutine crc_update_bytes

  subroutine build_crc_table(crc_table)
    integer(kind=8), intent(out) :: crc_table(0:255)
    integer(kind=8), parameter :: poly = int(z'EDB88320', 8)
    integer(kind=8) :: c
    integer :: n, k
    do n = 0, 255
       c = int(n, 8)
       do k = 1, 8
          if (iand(c, 1_8) == 1_8) then
             c = ieor(ishft(c, -1), poly)
          else
             c = ishft(c, -1)
          end if
       end do
       crc_table(n) = c
    end do
  end subroutine build_crc_table

  function char_be32(v) result(s)
    integer, intent(in) :: v
    character(len=4) :: s
    integer(kind=8) :: vv
    vv = int(v, 8)
    s = achar(int(iand(ishft(vv, -24), 255_8))) // &
        achar(int(iand(ishft(vv, -16), 255_8))) // &
        achar(int(iand(ishft(vv, -8),  255_8))) // &
        achar(int(iand(vv, 255_8)))
  end function char_be32

  function char_be32_8(v) result(s)
    integer(kind=8), intent(in) :: v
    character(len=4) :: s
    s = achar(int(iand(ishft(v, -24), 255_8))) // &
        achar(int(iand(ishft(v, -16), 255_8))) // &
        achar(int(iand(ishft(v, -8),  255_8))) // &
        achar(int(iand(v, 255_8)))
  end function char_be32_8

end program julia_fractal

 Saved julia256.png (         600 x         700 )


![Pendulum animation](julia256.png)